# Exploração raw — tancagem

Notebook unificado: **perfil exploratório**, **qualidade/chave lógica** e **inventário temporal** dos CSV brutos.

| Seção | Objetivo |
|-------|----------|
| 1. Perfil | Schema, tipos, nulos e domínios categóricos (snapshot recente) |
| 2. Qualidade | Duplicatas, chave primária candidata, CNPJ e agregação por instalação |
| 3. Piloto temporal | Inventário de todos os arquivos baixados (linhas, m³) |

**Pré-requisito:** `py estudos/tancagem-abastecimento/pipelines/download_raw.py`

In [ ]:
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _paths import RAW_DIR, REPO_ROOT

SAMPLE = RAW_DIR / "2026" / "janeiro.csv"
KEY = ["Data", "CodInstalacao", "Tag", "GrupoDeProdutos"]
CAT_COLS = ["Segmento", "DetalheInstalacao", "GrupoDeProdutos", "TipoDaUnidade", "Uf"]

if not SAMPLE.exists():
    raise FileNotFoundError(f"Execute download_raw.py. Ausente: {SAMPLE}")

df = pd.read_csv(SAMPLE, encoding="utf-8")
print(f"Snapshot: {SAMPLE.relative_to(REPO_ROOT)}")
print(f"Linhas: {len(df):,} | Colunas: {list(df.columns)}")

## 1. Perfil exploratório

In [ ]:
df.head(3)

In [ ]:
df.info()
df.describe(include="all").T

In [ ]:
for col in CAT_COLS:
    print(f"\n=== {col} ({df[col].nunique()} únicos) ===")
    print(df[col].value_counts(dropna=False).head(15).to_string())

In [ ]:
print("Data — distintos:", df["Data"].nunique())
print(df["Data"].value_counts().head())
print("\nTancagemM3 — min/max:", df["TancagemM3"].min(), df["TancagemM3"].max())
print(f"Soma nacional (snapshot): {df['TancagemM3'].sum():,.0f} m³")

## 2. Qualidade e chave lógica

Hipótese de chave: `Data` + `CodInstalacao` + `Tag` + `GrupoDeProdutos`

In [ ]:
n = len(df)
n_unique = df[KEY].drop_duplicates().shape[0]
dup = df[df.duplicated(KEY, keep=False)].sort_values(KEY)

print(f"Linhas: {n:,}")
print(f"Chaves únicas ({' + '.join(KEY)}): {n_unique:,}")
print(f"Duplicatas na chave: {n - n_unique:,}")
dup.head(10)

In [ ]:
print("Nulos por coluna:")
print(df.isna().sum())

cnpj_len = df["Cnpj"].astype(str).str.len()
print("\nCNPJ — comprimento (moda):", cnpj_len.mode().iloc[0])
print(cnpj_len.value_counts().head())

print("\nTancagemM3 <= 0:", (df["TancagemM3"] <= 0).sum())

In [ ]:
by_inst = df.groupby(["Data", "CodInstalacao"], as_index=False)["TancagemM3"].sum()
print("Instalações no snapshot:", len(by_inst))
print("Top 5 instalações por m³:")
by_inst.nlargest(5, "TancagemM3")

## 3. Piloto temporal (inventário de arquivos)

Diagnóstico por arquivo — orienta integração histórica. Não consolida a série.

In [ ]:
files = sorted(RAW_DIR.rglob("*.csv"))
print(f"{len(files)} arquivos CSV em {RAW_DIR.relative_to(REPO_ROOT)}")

In [ ]:
rows = []
for path in files:
    try:
        part = pd.read_csv(path, encoding="utf-8")
        rows.append(
            {
                "arquivo": str(path.relative_to(RAW_DIR)),
                "linhas": len(part),
                "colunas": len(part.columns),
                "data_distintas": part["Data"].nunique() if "Data" in part.columns else None,
                "soma_m3": part["TancagemM3"].sum() if "TancagemM3" in part.columns else None,
            }
        )
    except Exception as e:
        rows.append({"arquivo": str(path.relative_to(RAW_DIR)), "erro": str(e)})

inv = pd.DataFrame(rows)
inv

In [ ]:
other = [p for p in RAW_DIR.rglob("*") if p.is_file() and p.suffix.lower() not in {".csv"}]
for p in other:
    print(p.relative_to(RAW_DIR), f"({p.stat().st_size / 1024:.1f} KB)")